In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
 
print("Unsloth installed successfully!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-c7jj5u_3/unsloth_47d6ac1c3fac4a85a7c0e0604fda972e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-c7jj5u_3/unsloth_47d6ac1c3fac4a85a7c0e0604fda972e
  Resolved https://github.com/unslothai/unsloth.git to commit 46f9be3dd1673ce38d662c25de005e233056c21b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 

In [9]:
from datasets import load_dataset

datasets = load_dataset("json", data_files="/kaggle/input/datasets/wangzhi0201/datasets-extend-15k/extend_datasets_7_json.jsonl")
datasets = datasets
print(datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'scores', 'distribution', 'expected_value'],
        num_rows: 15402
    })
})


In [12]:
print(datasets["train"][1])

{'id': 1, 'text': 'Bước lên đỉnh cao với tấm HCV SEA Games năm 2009, nhưng số phận cơ thủ Đỗ Hoàng Quân là chuỗi những ngày tháng bất hạnh. Sau rất nhiều sóng gió cuộc đời, ông trời vẫn như muốn thử thách anh. Lần này là một biến cố lớn, khi nhà vô địch người Hà Nội với căn bệnh ung thư quái ác mới chỉ phát hiện cách đây ít ngày.', 'scores': {'coherence': {'distribution': [0.0, 0.0, 0.02, 0.08, 0.9], 'expected_value': 4.88}}, 'distribution': None, 'expected_value': None}


In [13]:
from unsloth import FastLanguageModel
from peft import PeftModel

import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/models/wangzhi0201/qlora-day2/pytorch/default/1",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map="auto"
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [14]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [19]:
from unsloth.chat_templates import get_chat_template

tokenizer_with_template = get_chat_template(
    tokenizer,
    chat_template="mistral",
    mapping={"role":"role","content":"content","user":"user","assistant":"assistant"},
    map_eos_token=True
)
import json

def format_datasets(example):
    # 1. Định nghĩa Instruction (System prompt)
    instruction = """You are an evaluator that measures the **coherence of a given text**.
Coherence should be judged based on four internal criteria:
* Grammar
* Fluency
* Relevance
* Consistency
However, **the final output must only report the coherence score**, not the individual criteria.
### Scoring Rubric
1 — Very Incoherent
2 — Mostly Incoherent
3 — Partially Coherent
4 — Mostly Coherent
5 — Fully Coherent

Your task is to return the evaluation in **JSON format** with the following structure:

* `distribution`: probability distribution over the 5 coherence levels `[p1, p2, p3, p4, p5]`
* `expected_value`: the expected coherence score computed from the distribution
### Example
Input:
"Máy in bị hết mực và tôi cần đi tắm ngay bây giờ. Nhận diện khuôn mặt rất nhanh nhưng tôi đang đeo khẩu trang kín."
Output:
{
    "scores": {
        "coherence": {
            "distribution": [0.3, 0.7, 0.0, 0.0, 0.0],
            "expected_value": 1.7
        }
    }
}

### Important Rules
* Return **only valid JSON**.
* Do **not include explanations or additional text**.
* Ensure the probability distribution sums to **1.0**.
 """
    user_content = f"{instruction}\n\nVăn bản cần đánh giá:\n{example['text']}"
    response_json = {
        "scores": example["scores"] 
    }
    assistant_content = json.dumps(response_json, ensure_ascii=False)

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content}
    ]
    
    full_text = tokenizer_with_template.apply_chat_template(messages, tokenize=False)
    
    return {"texts": full_text}
datasets = datasets.map(format_datasets)
# datasets = datasets.remove_columns(["scores"])


Map:   0%|          | 0/15402 [00:00<?, ? examples/s]

In [21]:
# datasets = datasets.remove_columns(["scores"])
print(datasets['train'][0]['texts'])

<s>[INST] You are an evaluator that measures the **coherence of a given text**.
Coherence should be judged based on four internal criteria:
* Grammar
* Fluency
* Relevance
* Consistency
However, **the final output must only report the coherence score**, not the individual criteria.
### Scoring Rubric
1 — Very Incoherent
2 — Mostly Incoherent
3 — Partially Coherent
4 — Mostly Coherent
5 — Fully Coherent

Your task is to return the evaluation in **JSON format** with the following structure:

* `distribution`: probability distribution over the 5 coherence levels `[p1, p2, p3, p4, p5]`
* `expected_value`: the expected coherence score computed from the distribution
### Example
Input:
"Máy in bị hết mực và tôi cần đi tắm ngay bây giờ. Nhận diện khuôn mặt rất nhanh nhưng tôi đang đeo khẩu trang kín."
Output:
{
    "scores": {
        "coherence": {
            "distribution": [0.3, 0.7, 0.0, 0.0, 0.0],
            "expected_value": 1.7
        }
    }
}

### Important Rules
* Return **only va

In [22]:
new_datasets = datasets
print(new_datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'scores', 'distribution', 'expected_value', 'texts'],
        num_rows: 15402
    })
})


In [6]:
model = PeftModel.from_pretrained(
    model,
    "/kaggle/input/models/wangzhi0201/qlora-model/pytorch/default/1/checkpoint-2838"
)
from trl import SFTTrainer
from transformers import TrainingArguments
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=new_datasets["train"],
#     dataset_text_field = "texts",
    
# )

NameError: name 'PeftModel' is not defined

In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [23]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=new_datasets["train"],
    dataset_text_field="texts",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=5,
        learning_rate=2e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=3,

        ddp_find_unused_parameters=False,   
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/15402 [00:00<?, ? examples/s]

In [24]:
!torchrun --nproc_per_node=2 train.py

W0316 15:23:34.046000 460 torch/distributed/run.py:803] 
W0316 15:23:34.046000 460 torch/distributed/run.py:803] *****************************************
W0316 15:23:34.046000 460 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0316 15:23:34.046000 460 torch/distributed/run.py:803] *****************************************
/usr/bin/python3: can't open file '/kaggle/working/train.py': [Errno 2] No such file or directory
/usr/bin/python3: can't open file '/kaggle/working/train.py': [Errno 2] No such file or directory
E0316 15:23:34.225000 460 torch/distributed/elastic/multiprocessing/api.py:882] failed (exitcode: 2) local_rank: 0 (pid: 466) of binary: /usr/bin/python3
Traceback (most recent call last):
  File "/usr/local/bin/torchrun", line 10, in <module>
    sys.exit(main())
             ^^^^^^


In [ ]:
#today
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 15,402 | Num Epochs = 3 | Total steps = 11,553
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,1.519961
2,0.690744
3,0.950952
4,0.326669
5,0.878020
6,1.145083
7,0.535932
8,1.199185
9,0.454977
10,1.028812


In [18]:
#last day
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 3,781 | Num Epochs = 3 | Total steps = 2,838
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,1.826750
2,1.831428
3,1.671762
4,1.609275
5,1.876814
6,1.693589
7,1.618493
8,1.570351
9,1.617782
10,1.612867


TrainOutput(global_step=2838, training_loss=0.2627621620402549, metrics={'train_runtime': 13453.4603, 'train_samples_per_second': 0.843, 'train_steps_per_second': 0.211, 'total_flos': 1.8316673645314867e+17, 'train_loss': 0.2627621620402549, 'epoch': 3.0})

In [24]:
import zipfile
import os

def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                
                # giữ cấu trúc thư mục
                arcname = os.path.relpath(file_path, folder_path)
                
                zipf.write(file_path, arcname)

# path adapter
folder_to_zip = "/kaggle/working/outputs/checkpoint-2838"
zip_file = "adapter_model_2.zip"

zip_folder(folder_to_zip, zip_file)

print("Zip completed:", zip_file)

Zip completed: adapter_model_2.zip


In [21]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096

#load model
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True
)

model.load_adapter("/kaggle/working/outputs/checkpoint-2838")

model.eval()

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
    

In [37]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel

model_name = "unsloth/mistral-7b-bnb-4bit"
lora_path = "/kaggle/working/outputs/checkpoint-2838"

# load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    device_map="auto"
)

# load LoRA
model = PeftModel.from_pretrained(model, lora_path)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [52]:
def generate_response(text):
    device = model.device
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

inp ="""[INST] Bạn là một nhà đánh giá coherence của văn bản.
    Coherence bạn có thể hiểu và đánh giá theo  4 tiêu chí : grammar , fluency , relevant , consistency .
RUBRIC:
1 — Very Incoherent
2 — Mostly Incoherent
3 — Partially Coherent
4 — Mostly Coherent
5 — Fully Coherent
Hãy trả về kết quả dưới định dạng JSON chứa các điểm số: distribution [p1,p2,p3,p4,p5] và expected_value.
Chỉ trả về JSON, không giải thích gì thêm.

Văn bản cần đánh giá:
Sau chiến tích vang dội dẫn dắt U23 Việt Nam thi đấu thành công tại VCK U23 châu Á, HLV Park Hang Seo được Cục trưởng Cục Hộ tịch, Quốc tịch và Chứng thực (Bộ Tư pháp) gợi ý nhập Quốc tịch Việt Nam.[/INST]"""
output = generate_response(inp)
print(output)

[INST] Bạn là một nhà đánh giá coherence của văn bản.
    Coherence bạn có thể hiểu và đánh giá theo  4 tiêu chí : grammar , fluency , relevant , consistency .
RUBRIC:
1 — Very Incoherent
2 — Mostly Incoherent
3 — Partially Coherent
4 — Mostly Coherent
5 — Fully Coherent
Hãy trả về kết quả dưới định dạng JSON chứa các điểm số: distribution [p1,p2,p3,p4,p5] và expected_value.
Chỉ trả về JSON, không giải thích gì thêm.

Văn bản cần đánh giá:
Sau chiến tích vang dội dẫn dắt U23 Việt Nam thi đấu thành công tại VCK U23 châu Á, HLV Park Hang Seo được Cục trưởng Cục Hộ tịch, Quốc tịch và Chứng thực (Bộ Tư pháp) gợi ý nhập Quốc tịch Việt Nam.[/INST] { "distribution": [ 4.0, 0.0, 0.0, 0.0, 0.0 ], "expected_value": 4.0 }


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)
model.load_adapter("outputs/checkpoint-2838")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=6,   # tiếp tục train thêm
        learning_rate=2e-5,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)
trainer.train(resume_from_checkpoint="outputs/checkpoint-2838")

In [13]:
def formatting_prompts_func(examples):
    """Format dataset for instruction fine-tuning."""
    if "mistral" in model_name.lower():
        texts = []
        for instruction, input_text, output in zip(
            examples["instruction"],
            examples["input"],
            examples["output"]
        ):
            text = f"<s>[INST] {instruction}\n\n{input_text} [/INST] {output}</s>"
            texts.append(text)
        return {"text": texts}
    elif "llama" in model_name.lower():
        texts = []
        for instruction, input_text, output in zip(
            examples["instruction"],
            examples["input"],
            examples["output"]
        ):
            text = f"""system
{instruction}
user
{input_text}
assistant
{output}"""
            texts.append(text)
        return {"text": texts}
    else:
        texts = []
        for instruction, input_text, output in zip(
            examples["instruction"],
            examples["input"],
            examples["output"]
        ):
            text = f"""### Instruction:
{instruction}
 
### Input:
{input_text}
 
### Response:
{output}"""
            texts.append(text)
        return {"text": texts}
 
print("Formatting dataset...")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset formatted: {len(dataset)} samples")

Formatting dataset...


NameError: name 'dataset' is not defined